# LLM baselines (GPT-4o-mini)

Zero-shot evaluation on the same task families used in:
- `notebooks/pipeline_tetris.ipynb` (rotation / mirror discrimination)
- `notebooks/pipeline_maze.ipynb` (maze generation + start/goal semantics)
- `notebooks/pipeline_3d.ipynb` (Ganis-Kievit 3D blocks pairs in `data/test_balanced.npy`)

This notebook calls the OpenAI API (vision) using `gpt-4o-mini`.

Prereqs:
- `pip install openai pillow numpy opencv-python tqdm`
- Set `OPENAI_API_KEY` in your environment.


In [7]:
from __future__ import annotations

import base64
import hashlib
import json
import math
import os
import random
import re
import time
from collections import deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import cv2
from PIL import Image, ImageDraw
from tqdm.auto import tqdm


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "benchmarks").exists() and (p / "notebooks").exists():
            return p
    return start


REPO_ROOT = find_repo_root(Path.cwd())
print("Repo root:", REPO_ROOT)

# --- Config (cost/latency controls) ---
MODEL = "gpt-4o-mini"
SEED = 0

# Match the pipeline defaults
IMG_SIZE = 64
MAZE_CELLS = 9

# Make images easier for the VLM without changing the underlying grid.
UPSCALE = 4  # 64 -> 256
PAIR_PAD = 12

# Sample sizes
N_TETRIS = 10
N_COLORED_SHAPES = 10
N_3D_BLOCKS = 10
N_MAZE_TRACE = 10
N_MAZE_SOLVE = 10

# Cache LLM calls to avoid re-paying for reruns.
CACHE_PATH = REPO_ROOT / "benchmarks" / "llm_baselines_cache.jsonl"

random.seed(SEED)
np.random.seed(SEED)


Repo root: /Users/masha/Documents/visual-reasoning


In [8]:
# --- OpenAI client + on-disk cache ---

try:
    from openai import OpenAI
except Exception as e:
    raise RuntimeError("Missing dependency: openai. Install with `pip install openai`.") from e

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY in your environment before running.")

client = OpenAI()


def _load_jsonl_cache(path: Path) -> Dict[str, Dict[str, Any]]:
    cache: Dict[str, Dict[str, Any]] = {}
    if not path.exists():
        return cache
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            cache[obj["key"]] = obj
    return cache


_CACHE = _load_jsonl_cache(CACHE_PATH)
print(f"Loaded cache: {len(_CACHE)} entries from {CACHE_PATH}")


def _pil_to_png_bytes(img: Image.Image) -> bytes:
    import io

    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


def _pil_to_data_url(img: Image.Image) -> str:
    b = _pil_to_png_bytes(img)
    return "data:image/png;base64," + base64.b64encode(b).decode("utf-8")


def _cache_key(model: str, prompt: str, image_bytes: bytes) -> str:
    h = hashlib.sha256()
    h.update(model.encode("utf-8"))
    h.update(b"\n")
    h.update(prompt.encode("utf-8"))
    h.update(b"\n")
    h.update(image_bytes)
    return h.hexdigest()


def _extract_output_text(resp: Any) -> str:
    # OpenAI Python SDK (1.x) provides `output_text` on Responses API objects.
    t = getattr(resp, "output_text", None)
    if isinstance(t, str) and t.strip():
        return t.strip()

    # Chat Completions fallback shape
    try:
        return resp.choices[0].message.content.strip()  # type: ignore[attr-defined]
    except Exception:
        pass

    return str(resp).strip()


def llm_vision(
    prompt: str,
    image: Image.Image,
    *,
    model: str = MODEL,
    max_output_tokens: int = 64,
    temperature: float = 0.0,
    use_cache: bool = True,
    retries: int = 5,
) -> str:
    img = image.convert("RGB")
    png = _pil_to_png_bytes(img)
    key = _cache_key(model, prompt, png)
    if use_cache and key in _CACHE:
        return _CACHE[key]["response"]

    data_url = "data:image/png;base64," + base64.b64encode(png).decode("utf-8")

    last_err: Optional[Exception] = None
    for attempt in range(retries + 1):
        try:
            # Preferred: Responses API
            resp = client.responses.create(
                model=model,
                input=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "input_text", "text": prompt},
                            {"type": "input_image", "image_url": data_url},
                        ],
                    }
                ],
                temperature=temperature,
                max_output_tokens=max_output_tokens,
            )
            text = _extract_output_text(resp)
            break
        except Exception as e:
            last_err = e
            # Fallback: Chat Completions (older call path)
            try:
                resp = client.chat.completions.create(
                    model=model,
                    messages=[
                        {
                            "role": "user",
                            "content": [
                                {"type": "text", "text": prompt},
                                {"type": "image_url", "image_url": {"url": data_url}},
                            ],
                        }
                    ],
                    temperature=temperature,
                    max_tokens=max_output_tokens,
                )
                text = _extract_output_text(resp)
                break
            except Exception as e2:
                last_err = e2
                if attempt >= retries:
                    raise
                time.sleep(2 ** attempt)
    else:
        raise last_err or RuntimeError("Unknown OpenAI error")

    if use_cache:
        entry = {
            "key": key,
            "ts": time.time(),
            "model": model,
            "prompt": prompt,
            "response": text,
        }
        CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
        with CACHE_PATH.open("a", encoding="utf-8") as f:
            f.write(json.dumps(entry) + "\n")
        _CACHE[key] = entry

    return text


Loaded cache: 0 entries from /Users/masha/Documents/visual-reasoning/benchmarks/llm_baselines_cache.jsonl


In [9]:
# --- Prompts (use these verbatim for evaluation) ---

ROTATION_PROMPT = (
    "Analyze the two objects in this image. Are they the exact same object just rotated, "
    "or are they different/mirrored? Answer strictly with one word: 'SAME' or 'DIFFERENT'."
)

MAZE_TRACE_PROMPT = (
    "Analyze this maze. Does the highlighted trace represent a valid, continuous path "
    "from the start to the goal without crossing any walls? Answer strictly with one word: 'YES' or 'NO'."
)

# Path-format prompt (you didn't specify an output format, so we enforce one to make scoring deterministic).
MAZE_SOLVE_PROMPT = (
    "Analyze this maze. Provide a valid path from the start to the goal without crossing any walls. "
    "Return ONLY a sequence of moves using the letters U, D, L, R (no spaces, no punctuation)."
)


def parse_choice(text: str, choices: Sequence[str]) -> Optional[str]:
    t = (text or "").strip().upper()
    m = re.search(r"[A-Z]+", t)
    if not m:
        return None
    tok = m.group(0)
    return tok if tok in choices else None


def extract_moves(text: str, max_len: int = 512) -> str:
    moves = re.findall(r"[UDLR]", (text or "").upper())
    return "".join(moves)[:max_len]


def _to_u8_from_minus1_1(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    x = (x + 1.0) * 127.5
    return np.clip(x, 0, 255).astype(np.uint8)


def _gray_to_rgb(img_u8: np.ndarray) -> np.ndarray:
    if img_u8.ndim == 3 and img_u8.shape[0] == 1:
        img_u8 = img_u8[0]
    if img_u8.ndim != 2:
        raise ValueError(f"Expected HxW (or 1xHxW), got shape={img_u8.shape}")
    return np.stack([img_u8, img_u8, img_u8], axis=-1)


def _resize_nearest(img: np.ndarray, scale: int) -> np.ndarray:
    h, w = img.shape[:2]
    return cv2.resize(img, (w * scale, h * scale), interpolation=cv2.INTER_NEAREST)


def make_pair_image(left_rgb_u8: np.ndarray, right_rgb_u8: np.ndarray, pad: int = PAIR_PAD) -> Image.Image:
    left = Image.fromarray(left_rgb_u8.astype(np.uint8), mode="RGB")
    right = Image.fromarray(right_rgb_u8.astype(np.uint8), mode="RGB")

    h = max(left.height, right.height)
    w = left.width + right.width + (3 * pad)
    out = Image.new("RGB", (w, h + 2 * pad), color=(255, 255, 255))
    out.paste(left, (pad, pad))
    out.paste(right, (2 * pad + left.width, pad))

    draw = ImageDraw.Draw(out)
    draw.text((pad + 2, 2), "A", fill=(0, 0, 0))
    draw.text((2 * pad + left.width + 2, 2), "B", fill=(0, 0, 0))
    return out


In [10]:
# --- Rotation datasets ---

CHIRAL_TETRIS_SHAPES = {
    "L": [(0, -1), (0, 0), (0, 1), (1, 1)],
    "J": [(0, -1), (0, 0), (0, 1), (-1, 1)],
    "S": [(0, 0), (1, 0), (0, 1), (-1, 1)],
    "Z": [(0, 0), (-1, 0), (0, 1), (1, 1)],
    "F": [(0, 0), (0, -1), (1, -1), (-1, 0), (0, 1)],
    "P": [(0, 0), (0, -1), (1, -1), (1, 0), (0, 1)],
}


def draw_tetris_shape_np(name: str, size: int = IMG_SIZE) -> np.ndarray:
    img = np.zeros((size, size), dtype=np.uint8)
    center = size // 2
    block_size = size // 8

    for dx, dy in CHIRAL_TETRIS_SHAPES[name]:
        x = center + (dx * block_size) - (block_size // 2)
        y = center + (dy * block_size) - (block_size // 2)
        cv2.rectangle(img, (x, y), (x + block_size, y + block_size), 255, -1)
    return img


def rotate_gray_pad_to_diag(img: np.ndarray, angle_deg: float) -> np.ndarray:
    h, w = img.shape
    diag = int(math.ceil(math.sqrt(h * h + w * w)))
    pad_h = max(0, diag - h)
    pad_w = max(0, diag - w)
    pad_top = pad_h // 2
    pad_bottom = pad_h - pad_top
    pad_left = pad_w // 2
    pad_right = pad_w - pad_left

    img_p = np.pad(img, ((pad_top, pad_bottom), (pad_left, pad_right)), mode="constant", constant_values=0)
    hp, wp = img_p.shape
    center = (wp / 2.0, hp / 2.0)
    M = cv2.getRotationMatrix2D(center, angle_deg, 1.0)
    rot = cv2.warpAffine(
        img_p,
        M,
        (wp, hp),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )

    y0 = (hp - h) // 2
    x0 = (wp - w) // 2
    return rot[y0 : y0 + h, x0 : x0 + w]


def sample_tetris_pair(rng: random.Random, *, mirror_prob: float = 0.5, angle_step: float = 5.0):
    key = rng.choice(list(CHIRAL_TETRIS_SHAPES.keys()))
    target = draw_tetris_shape_np(key, IMG_SIZE)

    mirrored = rng.random() < mirror_prob
    source = cv2.flip(target, 1) if mirrored else target.copy()

    angles = np.arange(0.0, 360.0, angle_step, dtype=np.float32)
    angle = float(rng.choice(list(angles)))
    source = rotate_gray_pad_to_diag(source, angle)

    gt = "DIFFERENT" if mirrored else "SAME"
    return source, target, gt


def random_colored_rectangles(h: int, w: int, num_shapes: int, rng: random.Random) -> np.ndarray:
    img = np.zeros((h, w, 3), dtype=np.float32)
    for _ in range(num_shapes):
        color = np.array([rng.random(), rng.random(), rng.random()], dtype=np.float32)
        y0 = rng.randint(0, h - 8)
        x0 = rng.randint(0, w - 8)
        y1 = min(h, y0 + rng.randint(6, max(7, h // 2)))
        x1 = min(w, x0 + rng.randint(6, max(7, w // 2)))
        img[y0:y1, x0:x1, :] = color
    return img


def rotate_rgb(img: np.ndarray, angle_deg: float) -> np.ndarray:
    h, w, _ = img.shape
    center = (w / 2.0, h / 2.0)
    M = cv2.getRotationMatrix2D(center, angle_deg, 1.0)
    return cv2.warpAffine(
        img,
        M,
        (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0.0, 0.0, 0.0),
    )


def sample_colored_shapes_pair(rng: random.Random, *, num_shapes: int = 4):
    img_a = random_colored_rectangles(IMG_SIZE, IMG_SIZE, num_shapes, rng)
    angle = float(rng.randint(0, 359))
    img_b = rotate_rgb(img_a, angle)

    same = rng.random() > 0.5
    if not same:
        img_b = cv2.flip(img_b, 1)

    gt = "SAME" if same else "DIFFERENT"
    return (img_a, img_b, gt)


def load_3d_blocks_pairs() -> List[Dict[str, Any]]:
    path = REPO_ROOT / "data" / "test_balanced.npy"
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Create it by running notebooks/pipeline_3d.ipynb or benchmarks/dinov3_3d_baseline.ipynb."
        )
    raw = np.load(str(path), allow_pickle=True)
    return list(raw)


def pair_image_from_gray_arrays(x0_u8: np.ndarray, x1_u8: np.ndarray) -> Image.Image:
    left = _resize_nearest(_gray_to_rgb(x0_u8), UPSCALE)
    right = _resize_nearest(_gray_to_rgb(x1_u8), UPSCALE)
    return make_pair_image(left, right, pad=PAIR_PAD)


def pair_image_from_rgb_float01(img0: np.ndarray, img1: np.ndarray) -> Image.Image:
    x0_u8 = np.clip(img0 * 255.0, 0, 255).astype(np.uint8)
    x1_u8 = np.clip(img1 * 255.0, 0, 255).astype(np.uint8)
    left = _resize_nearest(x0_u8, UPSCALE)
    right = _resize_nearest(x1_u8, UPSCALE)
    return make_pair_image(left, right, pad=PAIR_PAD)


In [11]:
# --- Maze dataset + scoring ---

ACTION_TO_DELTA = {
    "U": (-1, 0),
    "D": (1, 0),
    "L": (0, -1),
    "R": (0, 1),
}


def generate_maze(cells_w: int, cells_h: int, rng: random.Random) -> np.ndarray:
    # Perfect maze with DFS backtracking. Returns grid with 1=wall, 0=free.
    grid = np.ones((cells_h * 2 + 1, cells_w * 2 + 1), dtype=np.uint8)
    visited = np.zeros((cells_h, cells_w), dtype=bool)

    stack = [(0, 0)]
    visited[0, 0] = True
    grid[1, 1] = 0

    while stack:
        x, y = stack[-1]
        neighbors = []
        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < cells_w and 0 <= ny < cells_h and not visited[ny, nx]:
                neighbors.append((nx, ny, dx, dy))
        if neighbors:
            nx, ny, dx, dy = rng.choice(neighbors)
            grid[y * 2 + 1 + dy, x * 2 + 1 + dx] = 0
            grid[ny * 2 + 1, nx * 2 + 1] = 0
            visited[ny, nx] = True
            stack.append((nx, ny))
        else:
            stack.pop()
    return grid


def bfs_shortest_path(grid: np.ndarray, start: Tuple[int, int], goal: Tuple[int, int]) -> List[Tuple[int, int]]:
    h, w = grid.shape
    q = deque([start])
    prev = {start: None}

    while q:
        y, x = q.popleft()
        if (y, x) == goal:
            break
        for dy, dx in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            ny, nx = y + dy, x + dx
            if 0 <= ny < h and 0 <= nx < w and grid[ny, nx] == 0 and (ny, nx) not in prev:
                prev[(ny, nx)] = (y, x)
                q.append((ny, nx))

    if goal not in prev:
        return []

    path = []
    cur = goal
    while cur is not None:
        path.append(cur)
        cur = prev[cur]
    path.reverse()
    return path


def is_trace_valid(
    grid: np.ndarray,
    start: Tuple[int, int],
    goal: Tuple[int, int],
    trace: List[Tuple[int, int]],
) -> bool:
    if not trace:
        return False
    if trace[0] != start or trace[-1] != goal:
        return False
    h, w = grid.shape
    for (y, x) in trace:
        if not (0 <= y < h and 0 <= x < w):
            return False
        if grid[y, x] != 0:
            return False
    for (y0, x0), (y1, x1) in zip(trace, trace[1:]):
        if abs(y0 - y1) + abs(x0 - x1) != 1:
            return False
    return True


def render_maze(
    grid: np.ndarray,
    start: Tuple[int, int],
    goal: Tuple[int, int],
    trace: Optional[List[Tuple[int, int]]] = None,
) -> Image.Image:
    h, w = grid.shape
    img = np.ones((h, w, 3), dtype=np.uint8) * 255
    img[grid == 1] = 0

    if trace is not None:
        for (y, x) in trace:
            if 0 <= y < h and 0 <= x < w:
                img[y, x] = np.array([255, 50, 50], dtype=np.uint8)

    sy, sx = start
    gy, gx = goal
    img[sy, sx] = np.array([50, 220, 50], dtype=np.uint8)
    img[gy, gx] = np.array([50, 50, 220], dtype=np.uint8)

    img_big = _resize_nearest(img, UPSCALE)
    return Image.fromarray(img_big, mode="RGB")


def verify_moves(grid: np.ndarray, start: Tuple[int, int], goal: Tuple[int, int], moves: str) -> bool:
    pos = start
    h, w = grid.shape
    for m in moves:
        if m not in ACTION_TO_DELTA:
            return False
        dy, dx = ACTION_TO_DELTA[m]
        ny, nx = pos[0] + dy, pos[1] + dx
        if not (0 <= ny < h and 0 <= nx < w):
            return False
        if grid[ny, nx] != 0:
            return False
        pos = (ny, nx)
    return pos == goal


@dataclass
class MazeInstance:
    grid: np.ndarray
    start: Tuple[int, int]
    goal: Tuple[int, int]


def make_maze_instance(rng: random.Random) -> MazeInstance:
    grid = generate_maze(MAZE_CELLS, MAZE_CELLS, rng)
    start = (1, 1)
    goal = (grid.shape[0] - 2, grid.shape[1] - 2)
    return MazeInstance(grid=grid, start=start, goal=goal)


def make_trace_sample(rng: random.Random) -> Tuple[Image.Image, str]:
    inst = make_maze_instance(rng)
    path = bfs_shortest_path(inst.grid, inst.start, inst.goal)
    if not path:
        # very rare; regenerate
        return make_trace_sample(rng)

    valid = rng.random() < 0.5
    if valid:
        trace = path
        gt = "YES"
    else:
        mode = rng.choice(["prefix", "gap", "wall"])
        trace = list(path)
        if mode == "prefix":
            k = rng.randint(2, max(3, len(trace) - 2))
            trace = trace[:k]
        elif mode == "gap":
            if len(trace) > 6:
                i = rng.randint(2, len(trace) - 4)
                trace = trace[:i] + trace[i + 2 :]
            else:
                trace = trace[: max(2, len(trace) // 2)]
        else:  # wall
            inserted = False
            for _ in range(50):
                i = rng.randint(1, len(trace) - 2)
                y, x = trace[i]
                candidates = [(y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)]
                rng.shuffle(candidates)
                for ny, nx in candidates:
                    if 0 <= ny < inst.grid.shape[0] and 0 <= nx < inst.grid.shape[1] and inst.grid[ny, nx] == 1:
                        trace.insert(i + 1, (ny, nx))
                        inserted = True
                        break
                if inserted:
                    break
            if not inserted:
                trace = trace[: max(2, len(trace) // 2)]

        gt = "NO"

    # Sanity check labeling
    if is_trace_valid(inst.grid, inst.start, inst.goal, trace):
        gt = "YES"

    img = render_maze(inst.grid, inst.start, inst.goal, trace=trace)
    return img, gt


In [12]:
# --- Evaluation loops ---

def eval_rotation(
    name: str,
    images_and_labels: List[Tuple[Image.Image, str]],
    *,
    max_output_tokens: int = 8,
) -> Dict[str, Any]:
    results = []
    for img, gt in tqdm(images_and_labels, desc=f"{name} (rotation)"):
        raw = llm_vision(ROTATION_PROMPT, img, max_output_tokens=max_output_tokens)
        pred = parse_choice(raw, ["SAME", "DIFFERENT"])
        correct = (pred == gt)
        results.append({"gt": gt, "pred": pred, "raw": raw, "correct": correct})
    acc = float(np.mean([r["correct"] for r in results])) if results else 0.0
    return {"name": name, "n": len(results), "accuracy": acc, "results": results}


def eval_maze_trace(images_and_labels: List[Tuple[Image.Image, str]]) -> Dict[str, Any]:
    results = []
    for img, gt in tqdm(images_and_labels, desc="Maze trace (YES/NO)"):
        raw = llm_vision(MAZE_TRACE_PROMPT, img, max_output_tokens=8)
        pred = parse_choice(raw, ["YES", "NO"])
        correct = (pred == gt)
        results.append({"gt": gt, "pred": pred, "raw": raw, "correct": correct})
    acc = float(np.mean([r["correct"] for r in results])) if results else 0.0
    return {"name": "maze_trace", "n": len(results), "accuracy": acc, "results": results}


def eval_maze_solve(instances: List[MazeInstance]) -> Dict[str, Any]:
    results = []
    for inst in tqdm(instances, desc="Maze solve (path)"):
        img = render_maze(inst.grid, inst.start, inst.goal, trace=None)
        raw = llm_vision(MAZE_SOLVE_PROMPT, img, max_output_tokens=512)
        moves = extract_moves(raw, max_len=512)
        ok = verify_moves(inst.grid, inst.start, inst.goal, moves)
        results.append({"moves": moves, "raw": raw, "success": ok, "len": len(moves)})
    success = float(np.mean([r["success"] for r in results])) if results else 0.0
    return {"name": "maze_solve", "n": len(results), "success_rate": success, "results": results}


# --- Build deterministic samples (for reproducibility + caching) ---
rng = random.Random(SEED)

# Tetris
tetris_samples: List[Tuple[Image.Image, str]] = []
for _ in range(N_TETRIS):
    src, tgt, gt = sample_tetris_pair(rng)
    img = pair_image_from_gray_arrays(src, tgt)
    tetris_samples.append((img, gt))

# Colored Shapes
colors_samples: List[Tuple[Image.Image, str]] = []
for _ in range(N_COLORED_SHAPES):
    a, b, gt = sample_colored_shapes_pair(rng)
    img = pair_image_from_rgb_float01(a, b)
    colors_samples.append((img, gt))

# 3D Blocks (Ganis-Kievit)
raw_3d = load_3d_blocks_pairs()
rng.shuffle(raw_3d)
raw_3d = raw_3d[: min(N_3D_BLOCKS, len(raw_3d))]

blocks3d_samples: List[Tuple[Image.Image, str]] = []
for s in raw_3d:
    x0_u8 = _to_u8_from_minus1_1(np.asarray(s["x0"]))
    x1_u8 = _to_u8_from_minus1_1(np.asarray(s["x1"]))
    gt = "SAME" if s.get("label") == "same" else "DIFFERENT"
    img = pair_image_from_gray_arrays(x0_u8, x1_u8)
    blocks3d_samples.append((img, gt))

# Maze trace validity
maze_trace_samples: List[Tuple[Image.Image, str]] = []
for _ in range(N_MAZE_TRACE):
    maze_trace_samples.append(make_trace_sample(rng))

# Maze solve
maze_solve_instances: List[MazeInstance] = [make_maze_instance(rng) for _ in range(N_MAZE_SOLVE)]


# --- Run ---
tetris_res = eval_rotation("tetris", tetris_samples)
colors_res = eval_rotation("colored_shapes", colors_samples)
blocks3d_res = eval_rotation("3d_blocks", blocks3d_samples)

maze_trace_res = eval_maze_trace(maze_trace_samples)
maze_solve_res = eval_maze_solve(maze_solve_instances)


def pct(x: float) -> str:
    return f"{100.0 * x:.2f}%"


print("\n=== Summary (zero-shot) ===")
print(f"Rotation | tetris        : {pct(tetris_res['accuracy'])} (n={tetris_res['n']})")
print(f"Rotation | colored_shapes: {pct(colors_res['accuracy'])} (n={colors_res['n']})")
print(f"Rotation | 3d_blocks     : {pct(blocks3d_res['accuracy'])} (n={blocks3d_res['n']})")
print(f"Maze     | trace valid   : {pct(maze_trace_res['accuracy'])} (n={maze_trace_res['n']})")
print(f"Maze     | solve success : {pct(maze_solve_res['success_rate'])} (n={maze_solve_res['n']})")


Maze solve (path): 100%|██████████| 10/10 [00:28<00:00,  2.87s/it]


=== Summary (zero-shot) ===
Rotation | tetris        : 70.00% (n=10)
Rotation | colored_shapes: 40.00% (n=10)
Rotation | 3d_blocks     : 40.00% (n=10)
Maze     | trace valid   : 60.00% (n=10)
Maze     | solve success : 0.00% (n=10)
